# Food Waste Reducer - YOLO training (Colab, one click)

> **Data gate: OPEN** (ML QA PASS on #27, 2026-10-10: leak-free session/pHash split, stratification rules).
> Cell 4 still re-checks it on the data it builds here and REFUSES to train unless `split_grouping == session`,
> `ml/eval/leakage_check.py` passes (0 held-out images within 300 s of train/val; pHash near-dups <= 1% per source)
> and `ml/eval/stratification_check.py` reports no failures.
>
> **Checkpoint: `last.pt`, fixed epochs** (ML-28): val has no potato and no apple/spoiled, so "best" (picked on val)
> would be blind to them. Metrics are reported on the held-out split from the last epoch.

> **If you got disconnected: reconnect, keep the same `RUN_ID`, Runtime -> Run all; it resumes from the last epoch on Drive.**
> Checkpoints (`last.pt`, written every epoch) and the prepared dataset live in Google Drive under `MyDrive/fwr_runs/<RUN_ID>/`.
> Type your `RUN_ID` into the form field in cell 1 and **write it down**. Left at the default it becomes today's date, so a
> run resumed after midnight would start a fresh run instead. Finished stages are skipped, so re-running a completed notebook only re-exports.

## What the owner clicks
1. **Runtime -> Change runtime type -> T4 GPU.**
2. Left sidebar **Secrets** (key icon): add `ROBOFLOW_API_KEY` and `KAGGLE_API_TOKEN`; switch **Notebook access** on for each. (`KAGGLE_API_TOKEN` is the Kaggle access token, used as a Bearer token; optional: without it the Kaggle source is skipped.)
3. **Runtime -> Run all.** Allow Drive access when asked: **Drive is required** (cell 1 stops with a message if it is not mounted).
4. When it finishes, the last cell downloads **`export.zip`** (detector.pt, freshness.pt, class_names.json, predictions.jsonl, heldout_gt.jsonl, heldout_manifest.csv, dataset_report.json). Give it to the team.

Expect roughly 3-4.5 h on a free T4 for the full run (80 epochs yolo26s + 30 epochs classifier); free Colab disconnects, which is what the resume is for. Ultralytics is **AGPL-3.0**; see `docs/ML.md`. Never paste keys into cells.

In [ ]:
# 1. Setup: code, deps, Google Drive (required), run folder
import os, shutil, subprocess, sys
REPO_URL = "https://github.com/Abyss999/CodeRedOrion-CougarAIUncs.git"
if not os.path.isdir("/content/repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/repo"], check=True)
%cd /content/repo/ml/yolo
!pip install -q -r requirements.txt roboflow imagehash
!python -c "import ultralytics; print('ultralytics', ultralytics.__version__)"   # expect 8.4.175
import imagehash   # the pHash leakage gate needs it: fail loudly here, never skip silently
print("imagehash", imagehash.__version__)

# RUN_ID names the folder on Drive that holds checkpoints and the prepared dataset.
# KEEP IT THE SAME when you resume after a disconnect. Leave the placeholder to use today's date (then write it down).
RUN_ID = "fwr_YYYYMMDD"  #@param {type:"string"}
import datetime, re
if RUN_ID == "fwr_YYYYMMDD":
    RUN_ID = "fwr_" + datetime.date.today().strftime("%Y%m%d")
    print("RUN_ID was the placeholder, using today's date:", RUN_ID, "(write it down; type it in the form to resume after midnight)")
assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", RUN_ID), "RUN_ID must start with a letter or digit and contain only letters, digits, _ and - (not empty, not . or ..)"

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    raise SystemExit(f"STOP: Google Drive is required (checkpoints must survive a disconnect) and could not be mounted: {e!r}\n"
                     "Re-run this cell and approve the Drive permission prompt.")
if not os.path.isdir("/content/drive/MyDrive"):
    raise SystemExit("STOP: Google Drive is not mounted (/content/drive/MyDrive missing). Re-run this cell and approve the Drive prompt.")
RUNS_ROOT = "/content/drive/MyDrive/fwr_runs"
RUN_DIR = f"{RUNS_ROOT}/{RUN_ID}"          # train.py --project: det/weights/last.pt and cls/weights/last.pt are written here every epoch
os.makedirs(RUN_DIR, exist_ok=True)
print("existing runs on Drive:", sorted(os.listdir(RUNS_ROOT)))
print("RUN_DIR:", RUN_DIR, "| det checkpoint on Drive:", os.path.exists(f"{RUN_DIR}/det/weights/last.pt"),
      "| cls checkpoint on Drive:", os.path.exists(f"{RUN_DIR}/cls/weights/last.pt"))

# Prepared dataset cache on Drive. A single tar (the dataset is ~39k small files; copying them one by one through the Drive
# mount would take far longer than the tar). The stamp file is written LAST, so a half-written copy is never trusted.
import hashlib, json, subprocess, tarfile
DATA = "/content/fwr_data"
DRIVE_TAR, DRIVE_DATA, DRIVE_SHA = f"{RUN_DIR}/fwr_data.tar", f"{RUN_DIR}/fwr_data", f"{RUN_DIR}/fwr_data.sha256"
TOPDIRS = ("images", "labels", "freshness", "cls")
def _top(rel):
    p = rel.split("/")
    return p[0] if len(p) > 1 and p[0] in TOPDIRS else None
def dir_counts(root):
    """File counts per top-level dataset dir. Ultralytics' *.cache files are ignored."""
    c = dict.fromkeys(TOPDIRS, 0)
    for dp, _, fns in os.walk(root):
        for f in fns:
            t = _top(os.path.relpath(os.path.join(dp, f), root).replace(os.sep, "/"))
            if t and not f.endswith(".cache"):
                c[t] += 1
    return c
def tar_counts(tar_path):
    """Read the whole archive (fails on truncation): file counts per top-level dir (members are 'fwr_data/<dir>/...') + total members."""
    c, n = dict.fromkeys(TOPDIRS, 0), 0
    with tarfile.open(tar_path) as tf:
        for m in tf:
            n += 1
            t = _top(m.name.split("/", 1)[1]) if "/" in m.name else None
            if t and m.isfile():
                c[t] += 1
    return c, n
def make_stamp(report_path, tar_path, counts, members):
    return json.dumps({"report_sha256": hashlib.sha256(open(report_path, "rb").read()).hexdigest(),
                       "tar_bytes": os.path.getsize(tar_path), "tar_members": members, "counts": counts}, sort_keys=True)
def read_stamp():
    try:
        return json.load(open(DRIVE_SHA))
    except Exception:
        return None
_st = read_stamp()
DATA_ON_DRIVE = bool(_st and os.path.exists(DRIVE_TAR) and os.path.exists(f"{DRIVE_DATA}/dataset_report.json")
                     and _st["tar_bytes"] == os.path.getsize(DRIVE_TAR)
                     and _st["report_sha256"] == hashlib.sha256(open(f"{DRIVE_DATA}/dataset_report.json", "rb").read()).hexdigest())
print("prepared dataset on Drive, stamp OK -> skip downloads + prepare:" if DATA_ON_DRIVE else "no valid prepared dataset on Drive yet -> download + prepare:", DATA_ON_DRIVE)
if os.path.exists(DRIVE_SHA) and not DATA_ON_DRIVE:
    print("stale or invalid stamp on Drive, will rebuild")


In [ ]:
# 2. Secrets (Colab Secrets only; values are never printed)
from google.colab import userdata
def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None
for k in ("ROBOFLOW_API_KEY", "KAGGLE_API_TOKEN"):
    v = secret(k)
    if v: os.environ[k] = v
    print(k, "set" if v else "MISSING")
assert DATA_ON_DRIVE or os.environ.get("ROBOFLOW_API_KEY"), "Add ROBOFLOW_API_KEY in Colab Secrets (left sidebar) and re-run."
HAVE_KAGGLE = bool(os.environ.get("KAGGLE_API_TOKEN"))

In [ ]:
# 3. (skipped when the prepared dataset is already on Drive) Download the four sources and write a Colab sources.json derived from the repo's one
if DATA_ON_DRIVE:
    print('Prepared dataset found on Drive: skipping all downloads.')
else:
    import json, pathlib, zipfile, urllib.request
    RAW = pathlib.Path("/content/raw"); RAW.mkdir(exist_ok=True)
    srcs = {s["name"]: s for s in json.load(open("sources.json"))["sources"]}

    # 3a. VegQual (figshare, CC BY 4.0)
    vq = RAW / "vegqual"
    if not vq.exists():
        vq.mkdir()
        urllib.request.urlretrieve("https://ndownloader.figshare.com/files/62799448", "/content/vegqual.zip")
        zipfile.ZipFile("/content/vegqual.zip").extractall(vq)
        os.remove("/content/vegqual.zip")
    srcs["vegqual"]["root"] = str(next(vq.rglob("data.yaml")).parent)

    # 3b. Roboflow (YOLO txt export). Versions are pinned in sources.json (fridge_objects v1, hien_fridge v12).
    from roboflow import Roboflow
    rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
    for name in ("fridge_objects", "hien_fridge"):
        cfg = srcs[name]["roboflow"]
        dest = RAW / name
        if not dest.exists():
            rf.workspace(cfg["workspace"]).project(cfg["project"]).version(cfg["version"]).download("yolov8", location=str(dest))
        srcs[name]["root"] = str(dest); srcs[name].pop("pending", None)
        got = [l for l in open(dest / "data.yaml") if l.startswith("names")]
        print(name, "v%s" % cfg["version"], got)      # must match the class_map keys in sources.json

    # 3c. raghavrpotdar via the Kaggle REST API (Bearer token, CC0), only if the secret exists
    k = srcs["fresh_stale_fruits_veg"]
    if HAVE_KAGGLE:
        kd = RAW / "fresh_stale_fruits_veg"
        if not kd.exists():
            kd.mkdir()
            !curl -sL -H "Authorization: Bearer $KAGGLE_API_TOKEN" -o /content/kaggle.zip https://www.kaggle.com/api/v1/datasets/download/raghavrpotdar/fresh-and-stale-images-of-fruits-and-vegetables
            zipfile.ZipFile("/content/kaggle.zip").extractall(kd)
            os.remove("/content/kaggle.zip")
        k["root"] = str(next(kd.rglob("fresh_apple")).parent); k.pop("pending", None)
        print("kaggle class dirs:", sorted(p.name for p in pathlib.Path(k["root"]).iterdir() if p.is_dir()))
    else:
        k["pending"] = True
        print("KAGGLE_API_TOKEN missing: skipping fresh_stale_fruits_veg")

    json.dump({"sources": list(srcs.values())}, open("/content/sources.colab.json", "w"), indent=1)

In [ ]:
# 4. Build (or restore from Drive) the dataset and run the gates. Training REFUSES to start unless every gate passes.
import json
# The gates ALWAYS run, also on resume, on the restored copy of the Drive dataset. prepare_dataset.py is deterministic (seed 42,
# sha1-based group hashing, sorted traversal, seeded lighting noise) given the same pinned sources, but a restore reuses the exact bytes anyway.
if DATA_ON_DRIVE:
    shutil.rmtree(DATA, ignore_errors=True)
    subprocess.run(["tar", "-xf", DRIVE_TAR, "-C", "/content"], check=True)      # a failed extract raises; never ignored
    got = dir_counts(DATA)
    ok = (got == _st["counts"]
          and open(f"{DATA}/dataset_report.json", "rb").read() == open(f"{DRIVE_DATA}/dataset_report.json", "rb").read())
    if not ok:
        os.remove(DRIVE_SHA)       # invalidate the cache so the next Run all rebuilds it from scratch
        raise SystemExit(f"STOP: restored dataset does not match the Drive stamp (expected {_st['counts']}, got {got}). "
                         "The Drive cache was invalidated: Runtime -> Run all again to download and re-prepare.")
    print("restored from Drive:", got)
else:
    !python prepare_dataset.py --sources /content/sources.colab.json --out /content/fwr_data
rep = json.load(open("/content/fwr_data/dataset_report.json"))
assert rep.get("split_grouping") == "session", "Split is not session-grouped (ML-21). Stop: update the repo."
print("active classes:", rep["n_active_classes"], rep["active_classes"])
print("images:", {k: v["images"] for k, v in rep["splits"].items()})
print("dropped:", rep["dropped_images"])
print("near-duplicate drops:", rep["grouping"]["near_duplicate_drops"])
dec = rep["stratification"]["cell_decisions"]
not_meas = sorted(k for k, d in dec.items() if d["decision"] == "not_measurable_on_heldout")
print("not measurable on held-out:", {k: (dec[k]["images"], dec[k]["sessions"]) for k in not_meas})

# ML-21 gate: timestamp neighbours == 0 and pHash near-dups <= 1% per source (needs imagehash; exit code 1 on failure)
def run_check(script, json_path):
    if os.path.exists(json_path): os.remove(json_path)            # never read a stale result from an earlier or crashed run
    p = subprocess.run([sys.executable, script, DATA, "--json", json_path], capture_output=True, text=True)
    print(p.stdout[-3000:], p.stderr[-1500:])
    assert p.returncode == 0, f"{os.path.basename(script)} exited with {p.returncode}: gate failed or the check crashed"
    assert os.path.exists(json_path), f"{os.path.basename(script)} wrote no {json_path}"
run_check("/content/repo/ml/eval/leakage_check.py", "/content/leakage.json")
lk = json.load(open("/content/leakage.json"))["per_source"]
for s, v in lk.items():
    print(s, {x: v[x] for x in ("heldout_originals", "within_60s", "within_300s", "phash_near_dup", "phash_near_dup_pct")})
    assert v["criterion"]["timestamp_300s_zero"], f"{s}: held-out image within 300 s of a train/val image"
    assert v["criterion"]["phash_le_1pct"] is True, f"{s}: pHash near-duplicates > 1% (or the check was skipped)"

# ML-23/24 gate: stratification + report consistency
run_check("/content/repo/ml/eval/stratification_check.py", "/content/strat.json")
st = json.load(open("/content/strat.json"))
assert not st["never_trained_cells"], st["never_trained_cells"]
assert not st["failures"], st["failures"]  # >=50% train, not-measurable consistency, required cells (QA rules)
assert all(v["ok"] for v in st["report_vs_manifest"].values()), "dataset_report.json disagrees with the manifest/disk"
unexpected = sorted(set(st["missing_from_heldout_required"]) - set(not_meas))
assert not unexpected, f"required cells missing from held-out: {unexpected}"
print("ALL GATES PASSED")

# Cache the gated dataset on Drive for resume (only after the gates passed). The sha file is written last = commit marker.
if not DATA_ON_DRIVE:
    os.makedirs(DRIVE_DATA, exist_ok=True)
    if os.path.exists(DRIVE_SHA): os.remove(DRIVE_SHA)
    part = DRIVE_TAR + ".part"
    try:
        subprocess.run(["tar", "--exclude=*.cache", "-cf", part, "-C", "/content", "fwr_data"], check=True)   # disk full / quota / I/O raises
        want, got = dir_counts(DATA), None
        got, members = tar_counts(part)                      # re-reads the archive: a truncated tar raises here
        assert got == want, f"archive file counts {got} differ from the dataset {want}"
    except BaseException:
        if os.path.exists(part): os.remove(part)
        raise
    os.replace(part, DRIVE_TAR)
    for f in ("dataset_report.json", "data.yaml", "class_names.json", "heldout_manifest.csv"):   # small, readable copy beside the tar
        shutil.copy(f"{DATA}/{f}", f"{DRIVE_DATA}/{f}")
    open(DRIVE_SHA, "w").write(make_stamp(f"{DRIVE_DATA}/dataset_report.json", DRIVE_TAR, got, members) + "\n")   # LAST = commit marker
    print("dataset cached on Drive:", DRIVE_TAR, round(os.path.getsize(DRIVE_TAR) / 1e6), "MB", got)


In [ ]:
# 5. Train (detector 80 ep yolo26s, classifier 30 ep; exports last.pt, see ML-28) and export predictions.
# --resume: finished stages are skipped, an interrupted stage continues from <RUN_DIR>/<det|cls>/weights/last.pt (saved to Drive every epoch).
# To resume after a disconnect: reconnect, keep RUN_ID, Runtime -> Run all.
!python train.py --data /content/fwr_data --epochs 80 --cls-epochs 30 --device auto --weights last --resume --project "{RUN_DIR}"


In [ ]:
# 6. Quick held-out check, zip the export (it is already on Drive in RUN_DIR/export)
from ultralytics import YOLO
EXPORT = f"{RUN_DIR}/export"
m = YOLO(f"{EXPORT}/detector.pt")
print(m.val(data="/content/fwr_data/data.yaml", split="test"))   # last.pt on held-out (incl. synthetic low/warm copies; only real low/warm counts for lighting, ML-17/22). The real gate run is ml/eval on predictions.jsonl
!rm -rf /content/export_out && mkdir -p /content/export_out && cp -r "{EXPORT}" /content/export_out/export
!cd /content/export_out && zip -r export.zip export
from google.colab import files
files.download("/content/export_out/export.zip")
